# Semantic Model Test Harness for DAX (Unit & Regression Testing)

**Author:** vestergaardj  
**Date:** April 7, 2026

---

## Abstract

This notebook is a comprehensive, production-grade test harness for validating DAX measures in Microsoft Fabric semantic models using Semantic Link Labs. It enables BI developers, data engineers, and analysts to define, execute, and report on DAX test cases with full automation and traceability. The approach brings software engineering rigor to BI, supporting CI/CD workflows, regression detection, and business logic validation.

**Key Features:**
- Automated DAX test execution for any semantic model in Fabric
- Parameterized test cases: measure, filter context, expected value
- Robust DAX query generation and error handling
- Diagnostics for model structure and measure validation
- Detailed result reporting, including pass/fail, actual vs. expected, and DAX query trace
- Extensible for CI pipelines and regression analysis

**Intended Audience:**
- Power BI/Fabric semantic model developers
- Data engineers and analytics engineers
- QA/test automation for BI
- Anyone seeking to apply software testing best practices to DAX

---

## How to Use This Notebook

1. **Configure Environment:**
   - Ensure Semantic Link Labs (`sempy_labs`) is installed (auto-installs if missing).
   - Set your Fabric dataset and workspace names in the configuration cell.

2. **Define Test Cases:**
   - Edit the DAX test DataFrame to specify the measure, filter context, and expected value for each test.
   - Use GUIDs for keys (e.g., WorkspaceId) as shown in diagnostics.

3. **Run DAX Tests:**
   - Execute the test harness cell to run all test cases.
   - The harness will try multiple DAX query formats and print each attempt.

4. **Review Results:**
   - Results are displayed in a DataFrame with pass/fail, actual/expected, and the DAX query used.
   - Failed tests are highlighted for review.

5. **Diagnose Model Structure:**
   - Use the diagnostics cell to list available columns and test measures in your model.
   - Adjust test cases as needed based on actual model structure.

6. **Integrate with CI/CD:**
   - The notebook is designed for automation and can be integrated into CI pipelines for regression testing.

---



## 1. Import Required Libraries

This cell imports the core Python libraries used throughout the notebook: `os` for environment access, `datetime` for timestamping metadata, `pandas` for structuring test cases and results as DataFrames, and `IPython.display` for rendering rich output such as Markdown and styled tables.

It also runs `%pip install semantic-link-labs` to ensure the Semantic Link Labs package is available in the current environment. This is the library that provides the `evaluate_dax_impersonation` function used later to execute DAX queries against a Fabric semantic model. If the package is already installed, the command is a no-op; if not, it will be downloaded and installed automatically, making the notebook portable across environments.

In [ ]:
# Section 1: Import Required Libraries
import os
import datetime
import pandas as pd
from IPython.display import display, Markdown

%pip install semantic-link-labs

## 2. Notebook Metadata

This cell defines a metadata dictionary that captures the notebook's author, creation date, and title. The creation date is set dynamically using `datetime.date.today()` so it always reflects when the notebook was last executed, which is valuable for audit trails and versioning. The metadata is displayed as a dictionary for quick reference, and can be extended with additional fields (e.g., version, description, or environment details) as needed for traceability in CI/CD or team workflows.

In [ ]:
# Section 2: Define Notebook Metadata
notebook_metadata = {
    "author": "vestergaardj",
    "created": datetime.date.today().isoformat(),
    "title": "Semantic Model Test Harness for DAX"
}
display(notebook_metadata)

## 3. Define DAX Test Cases

This cell constructs a pandas DataFrame containing the DAX test cases that will be executed against the semantic model. Each row represents one test and has three columns:

- **`measure`** — The name of the DAX measure to evaluate (e.g., `# Reports`, `# Workspace Users`).
- **`filter_context`** — A DAX boolean expression that sets the filter context for the evaluation, simulating a specific slicer or page filter (e.g., filtering to a particular workspace).
- **`expected_value`** — The known-correct result that the measure should return under the given filter context. This is what the harness will compare against.

To add more tests, simply append additional dictionaries to the list inside `pd.DataFrame([...])`. The DataFrame is displayed at the end of the cell so you can visually verify the test definitions before execution.

In [ ]:
# Section 3: DAX Test Case Input
import pandas as pd

# Deep-dive DAX test cases for CatMan BI Tenant Stats semantic model
# These tests use specific filters and scenarios for more granular validation
dax_tests = pd.DataFrame([
    # Catalog - Report: # Reports for a specific workspace
    {"measure": "# Reports", "filter_context": "'Catalog - Report'[Report Workspace] = 'Arla DK'", "expected_value": 61},

    # # Catalog - Workspace: " Workspace Users for a specific workspace
    {"measure": "# Workspace Users", "filter_context": "'Catalog - Workspace'[Workspace] = 'CatMan Next DK - Demo'", "expected_value": 15},

    # Total workspaces (unfiltered, validates the full workspace catalog load)
    {"measure": "# Workspaces", "filter_context": "", "expected_value": 4147},

    # Total datasets in a known workspace (tests cross-table relationship)
    {"measure": "# DataSets", "filter_context": "'Catalog - Workspace'[Workspace] = 'Operations'", "expected_value": 7},

    # Total distinct users across all activity (validates Activity table grain)
    {"measure": "# Distinct Users", "filter_context": "", "expected_value": 3093},

    # Workspace content count for a known workspace (composite measure: reports + datasets + dataflows + dashboards)
    {"measure": "# Workspace Content", "filter_context": "'Catalog - Workspace'[Workspace] = 'Arla DK'", "expected_value": 76},

    # Total dashboards (unfiltered, validates dashboard catalog and ISFILTERED guard logic)
    {"measure": "# Dashboards", "filter_context": "", "expected_value": 165},
])
display(dax_tests)

## 4. Run DAX Tests

This is the core execution cell of the harness. It performs the following steps:

1. **Configuration** — Sets `DATASET_NAME` and `WORKSPACE_NAME`, which identify the Fabric semantic model to query. Update these values to point to your own model before running.

2. **Helper functions** — Two utility functions are defined:
   - `fix_filter_context_quotes()` converts single-quoted string values in filter expressions to double quotes, which is required by DAX syntax (single quotes are reserved for table names).
   - `is_boolean_filter()` detects whether a filter context string is a simple boolean expression (e.g., `'Table'[Column] = "Value"`) versus a table function like `FILTER(...)` or `VALUES(...)`, so the harness can construct the correct `CALCULATE` syntax.

3. **Test loop** — The cell iterates over every row in the `dax_tests` DataFrame. For each test case it:
   - Builds a `EVALUATE ROW(...)` DAX query wrapping the measure in a `CALCULATE` with the appropriate filter context.
   - Sends the query to the semantic model via `sempy_labs.evaluate_dax_impersonation()`.
   - Compares the returned value to the expected value and records pass/fail.
   - Catches any exceptions (e.g., connectivity errors, invalid DAX) and logs them as failures with the error message.

4. **Output** — All results are collected into `results_df` and displayed as a DataFrame showing measure, filter context, expected vs. actual value, pass/fail status, and the exact DAX query used — providing full traceability for debugging.


In [ ]:
# Section 4: Run DAX Tests
import sempy_labs
results = []
# Set your dataset and workspace names here
DATASET_NAME = "<INSERT_DATASET_NAME>"  # Update as needed
WORKSPACE_NAME = "<INSERT_WORKSPACE_NAME>"    # Update as needed

def fix_filter_context_quotes(filter_ctx):
    # Replace single quotes around string values with double quotes for DAX
    import re
    return re.sub(r"=\s*'([^']+)'", r'= "\1"', filter_ctx)

def is_boolean_filter(filter_ctx):
    # Returns True if filter_ctx is a boolean expression (e.g., 'Table'[Col] = 'val')
    import re
    # Exclude if starts with VALUES, ALL, FILTER, etc.
    table_funcs = ['VALUES', 'ALL', 'FILTER', 'SELECTCOLUMNS', 'SUMMARIZE', 'ADDCOLUMNS']
    if any(filter_ctx.strip().upper().startswith(f) for f in table_funcs):
        return False
    # Look for boolean operators
    return bool(re.search(r'(=|<>|<=|>=|<|>)', filter_ctx))

for idx, row in dax_tests.iterrows():
    measure = row['measure']
    filter_ctx = row['filter_context']
    expected = row['expected_value']
    try:
        dax_expr = measure if measure.startswith("=") else f"[{measure}]"
        if filter_ctx and filter_ctx.strip():
            filter_arg = fix_filter_context_quotes(filter_ctx)
            if is_boolean_filter(filter_arg):
                # Pass boolean expr directly to CALCULATE
                dax_query = f"EVALUATE ROW(\"Value\", CALCULATE({dax_expr}, {filter_arg}))"
                print(f"Trying DAX (boolean filter): {dax_query}")
            else:
                # Table expression, pass as-is
                dax_query = f"EVALUATE ROW(\"Value\", CALCULATE({dax_expr}, {filter_arg}))"
                print(f"Trying DAX (table expr): {dax_query}")
            df = sempy_labs.evaluate_dax_impersonation(
                dataset=DATASET_NAME,
                dax_query=dax_query,
                workspace=WORKSPACE_NAME
)
            result = df.iloc[0, 0] if not df.empty else None
            passed = (expected is not None and result == expected)
            results.append({
                'measure': measure,
                'filter_context': filter_ctx,
                'expected': expected,
                'actual': result,
                'pass': passed if expected is not None else None,
                'dax_query': dax_query
            })
        else:
            dax_query = f"EVALUATE ROW(\"Value\", {dax_expr})"
            print(f"Trying DAX (no filter): {dax_query}")
            df = sempy_labs.evaluate_dax_impersonation(
                dataset=DATASET_NAME,
                dax_query=dax_query,
                workspace=WORKSPACE_NAME
)
            result = df.iloc[0, 0] if not df.empty else None
            passed = (expected is not None and result == expected)
            results.append({
                'measure': measure,
                'filter_context': filter_ctx,
                'expected': expected,
                'actual': result,
                'pass': passed if expected is not None else None,
                'dax_query': dax_query
            })
    except Exception as e:
        results.append({
            'measure': measure,
            'filter_context': filter_ctx,
            'expected': expected,
            'actual': str(e),
            'pass': False,
            'dax_query': dax_query if 'dax_query' in locals() else None
        })
results_df = pd.DataFrame(results)
display(results_df)

## 5. Results Summary and Regression Highlight

This cell aggregates the test results from the previous step into a concise summary. It counts the total number of tests executed, how many passed, and how many failed.

If any tests failed, the cell renders a highlighted table of only the failed rows so they stand out immediately for review — this is the key output for regression detection. Each failed row shows the measure, filter context, expected value, actual value, and the DAX query that was executed, giving you everything needed to diagnose the discrepancy.

If all tests passed, a success message is displayed instead. This summary is designed for both interactive review during development and automated consumption in CI/CD pipelines where a non-zero failure count can trigger alerts or block deployments.

In [ ]:
# Section 5: Results Summary and Regression Highlight
num_tests = len(results_df)
num_passed = results_df['pass'].sum()
num_failed = num_tests - num_passed

print(f"Total tests: {num_tests}")
print(f"Passed: {num_passed}")
print(f"Failed: {num_failed}")

if num_failed > 0:
    display(Markdown("**❌ Failed Tests:**"))
    display(results_df[~results_df['pass']])
else:
    display(Markdown("**✅ All tests passed!**"))